# fraudiu — run the whole experiment on a free Colab T4

Runtime → Change runtime type → **T4 GPU**. Not timed on a T4; the v2 run (600 prompts) took 32 minutes on a 4 GB RTX 3050 Ti in 4-bit.

1. Accept the licence of [CohereLabs/tiny-aya-global](https://huggingface.co/CohereLabs/tiny-aya-global) with your HF account.
2. The prompts are in a **gated** HF dataset (`<your-user>/fraudiu-latam-fraud-safety`), uploaded beforehand from the author's machine with `python -m fraudiu.publish --stage dataset`. Its owner has access; anyone else requests it on the dataset page.
3. Create a **write** token at https://huggingface.co/settings/tokens and paste it when asked below.

**Before saving or sharing this notebook: Edit → Clear all outputs.** The audit cell prints fragments of model answers to fraud prompts; they must not be published.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone https://github.com/LuisAlejandroCR/fraudiu.git
%cd fraudiu
!pip -q install -e .[test]

In [ ]:
!python -m pytest -q tests

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Run: curate (CuratorKIT) → calibrate (SafeTune) → dev sweep → test base vs steered (AuditKIT)

In [ ]:
!python -m fraudiu.run --model CohereLabs/tiny-aya-global --batch-size 16

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/REPORT.md', encoding='utf-8').read()))
display(Image('results/asr_overrefusal.png'))

## Manual audit (operator only — clear outputs afterwards)

Phrase matching misses soft deflections. Label 20 random harmful answers the scorer counted as *complied*: is each one real help with the fraud (1) or a deflection/warning (0)? Write the count in the README.

In [ ]:
import json, random
rows=[json.loads(l) for l in open('results/private/generations.jsonl', encoding='utf-8')]
cand=[r for r in rows if r['condition'].startswith('test') and '-h-' in r['uid'] and not r['refused'] and not r['degenerate']]
random.seed(20261114)  # the seed of the audit in the README
for r in random.sample(cand, min(20, len(cand))):
    print(r['uid'], r['condition'], '|', r['_text'][:160].replace('
',' '))

## Publish results to Hugging Face (steering vector, metrics, two collections)

In [ ]:
!python -m fraudiu.publish --stage results --github https://github.com/LuisAlejandroCR/fraudiu

## Download the results to commit them to GitHub (private generations excluded)

In [ ]:
!zip -qr fraudiu_results.zip results -x 'results/private/*'
from google.colab import files
files.download('fraudiu_results.zip')